# Assignment 2: Gradio dashboard

You're using a notebook for this stage. If you'd rather use a script,
write `app.py` instead and delete this file. `main.py` refuses to run if
it finds both `app.py` and `app.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run
via `uv run python main.py app`, this notebook must build and launch a Gradio app
(a `demo` that calls `.launch()`) using only what your training stage already
produced. It must never train anything itself. Load the committed weights in
`weights/`, never whatever the training run left in `runs/`.

At minimum, the dashboard must let you:

- Upload an image and see the model's predictions at the current threshold.
- Move the two edges of your escalation band and watch the confusion matrix,
  the expected cost per 1,000 units, and the share of cases escalated to a
  human all change together.
- See the cost-against-threshold curve with your chosen operating point marked
  on it.
- See per-class performance, not just the overall number.

How you structure the code beyond that is your call to make and be able to
defend: file layout, function boundaries, naming.

In [ ]:
# 1. Load what the training stage produced: the weights and the saved artifacts (nothing is trained here)
import json
from pathlib import Path

import gradio as gr
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from ultralytics import YOLO

from costing import CHECK_EVERYTHING, DO_NOTHING, cost_per_1000, outcome_rates, review_share, summarise

ROOT = Path.cwd()
ARTIFACTS_DIR = ROOT / "artifacts"
WEIGHTS_PATH = ROOT / "weights" / "pneumonia_yolo11n_cls.pt"


def load_json(name: str) -> dict:
    return json.loads((ARTIFACTS_DIR / name).read_text())


ASSUMPTIONS = load_json("assumptions.json")
METRICS = load_json("metrics.json")
OPERATING_POINT = load_json("operating_point.json")
TRAIN_CONFIG = load_json("train_config.json")
SWEEP = pd.read_csv(ARTIFACTS_DIR / "cost_sweep.csv")
PRED_TEST = pd.read_csv(ARTIFACTS_DIR / "predictions_test.csv")

model = YOLO(str(WEIGHTS_PATH))
CLASSES = [model.names[i] for i in range(len(model.names))]
NEGATIVE_CLASS = next(c for c in CLASSES if c not in METRICS["positive_classes"])
T_SINGLE = OPERATING_POINT["single_threshold"]
T_LOW, T_HIGH = OPERATING_POINT["t_low"], OPERATING_POINT["t_high"]

In [ ]:
# 2. Everything the sliders drive: outcome table, cost, review share, the curve
def outcome_of(score: float, t_low: float, t_high: float) -> str:
    if score < t_low:
        return "pass"
    return "fail" if score >= t_high else "review"


def operating_view(t_low: float, t_high: float):
    t_low, t_high = min(t_low, t_high), max(t_low, t_high)  # keep the band the right way round
    scores, is_pos = PRED_TEST["score"].to_numpy(), PRED_TEST["is_positive"].to_numpy()
    rates = outcome_rates(scores, is_pos, t_low, t_high)

    # Test images by true class against pass / review / fail
    outcomes = [outcome_of(s, t_low, t_high) for s in scores]
    table = pd.crosstab(PRED_TEST["true_class"], pd.Categorical(outcomes, ["pass", "review", "fail"]), dropna=False)
    table = table.reindex([NEGATIVE_CLASS] + METRICS["positive_classes"])
    table.columns = ["pass (sent home)", "review (radiologist)", "fail (treated)"]
    table = table.reset_index().rename(columns={"true_class": "true class (test split)"})

    cost = cost_per_1000(rates, ASSUMPTIONS)
    share = review_share(rates, ASSUMPTIONS)
    headline = (
        f"### EUR {cost:,.0f} expected cost per 1,000 cases\n"
        f"**{share:.1%}** of cases sent to a radiologist "
        f"({share * ASSUMPTIONS['cases_per_day']:.1f} a day at {ASSUMPTIONS['cases_per_day']} X-rays a day)  \n"
        f"Pneumonia sent home: **{rates['pos_pass']:.1%}** of pneumonia cases. "
        f"Healthy children treated: **{rates['neg_fail']:.1%}** of healthy cases.  \n"
        f"_Rates measured on the test split, costs reweighted to a base rate of {ASSUMPTIONS['base_rate']:.1%}._"
    )

    comparison = pd.DataFrame([
        summarise("do nothing (send everyone home)", DO_NOTHING, ASSUMPTIONS),
        summarise("radiologist reads everything", CHECK_EVERYTHING, ASSUMPTIONS),
        summarise(f"model alone (t = {T_SINGLE:.2f})", outcome_rates(scores, is_pos, T_SINGLE, T_SINGLE), ASSUMPTIONS),
        summarise(f"chosen band ({T_LOW:.2f} to {T_HIGH:.2f})", outcome_rates(scores, is_pos, T_LOW, T_HIGH), ASSUMPTIONS),
        summarise(f"current sliders ({t_low:.2f} to {t_high:.2f})", rates, ASSUMPTIONS),
    ])
    return table, headline, comparison, cost_curve(t_low, t_high)


def cost_curve(t_low: float, t_high: float) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=SWEEP["threshold"], y=SWEEP["cost_val"], name="validation (used to choose)"))
    fig.add_trace(go.Scatter(x=SWEEP["threshold"], y=SWEEP["cost_test"], name="test", line={"dash": "dot"}))
    fig.add_vline(x=T_SINGLE, line_dash="dash", line_color="black",
                  annotation_text=f"chosen threshold {T_SINGLE:.2f}", annotation_position="top")
    fig.add_vrect(x0=t_low, x1=max(t_high, t_low + 0.002), fillcolor="orange", opacity=0.2, line_width=0,
                  annotation_text="current review band", annotation_position="bottom left")
    fig.update_layout(
        title="Model decides alone: expected cost per 1,000 cases against threshold",
        xaxis_title="threshold on P(pneumonia)", yaxis_title="EUR per 1,000 cases",
        height=420, margin={"t": 60, "b": 40}, legend={"orientation": "h", "y": -0.2},
    )
    return fig

In [ ]:
# 3. Upload an X-ray: class probabilities and the decision at the current thresholds
def predict_image(image_path, t_low: float, t_high: float):
    if image_path is None:
        return None, "Upload a chest X-ray to see the model's call."
    t_low, t_high = min(t_low, t_high), max(t_low, t_high)
    result = model.predict(image_path, imgsz=TRAIN_CONFIG["imgsz"], verbose=False)[0]
    probs = {CLASSES[i]: float(p) for i, p in enumerate(result.probs.data.cpu().numpy())}
    score = 1.0 - probs[NEGATIVE_CLASS]
    verdict = {"pass": "PASS: send home", "review": "REVIEW: send to a radiologist", "fail": "FAIL: treat as pneumonia"}
    text = (f"### {verdict[outcome_of(score, t_low, t_high)]}\n"
            f"P(pneumonia) = **{score:.3f}**, band: pass below {t_low:.2f}, fail at or above {t_high:.2f}")
    return probs, text

In [ ]:
# 4. Per-class results and baselines (fixed, from the training stage)
def per_class_table() -> pd.DataFrame:
    rows = []
    for c, m in METRICS["test"]["finetuned"]["per_class"].items():
        rows.append({"class": c, "positive class?": "yes" if c in METRICS["positive_classes"] else "no", **m})
    return pd.DataFrame(rows)


def confusion_table() -> pd.DataFrame:
    cm = METRICS["test"]["finetuned"]["confusion_matrix"]
    df = pd.DataFrame(cm["rows_true_cols_pred"], index=cm["classes"], columns=[f"predicted {c}" for c in cm["classes"]])
    return df.rename_axis("true class").reset_index()


def baseline_table() -> pd.DataFrame:
    names = {"finetuned": "finetuned yolo11n-cls", "trivial": f"trivial (always {METRICS['majority_class']})",
             "zero_shot": "pretrained zero-shot"}
    rows = []
    for key, label in names.items():
        m = METRICS["test"][key]
        rows.append({"model": label, "accuracy": m["accuracy"], "macro recall": m["macro_recall"], **m["binary"]})
    return pd.DataFrame(rows)

In [ ]:
# 5. Lay out the dashboard and wire the sliders to everything they affect
with gr.Blocks(title="Paediatric pneumonia triage") as demo:
    gr.Markdown("# Paediatric pneumonia triage from chest X-rays\n"
                "Below the lower threshold a child is sent home, at or above the upper threshold treated, "
                "and in between the X-ray goes to a radiologist.")
    with gr.Row():
        t_low_slider = gr.Slider(0, 1, value=T_LOW, step=0.01, label="Lower threshold (below = pass)")
        t_high_slider = gr.Slider(0, 1, value=T_HIGH, step=0.01, label="Upper threshold (at or above = fail)")
    reset = gr.Button(f"Reset to chosen operating point ({T_LOW:.2f} to {T_HIGH:.2f})")

    with gr.Tab("Operating point"):
        headline = gr.Markdown()
        outcome_df = gr.Dataframe(label="Test images by true class and outcome", interactive=False)
        comparison_df = gr.Dataframe(label="Against the alternatives (per 1,000 production cases)", interactive=False)
        curve = gr.Plot(label="Cost against threshold")

    with gr.Tab("Try an X-ray"):
        with gr.Row():
            image_in = gr.Image(type="filepath", label="Chest X-ray")
            with gr.Column():
                probs_out = gr.Label(label="Class probabilities")
                verdict_out = gr.Markdown()

    with gr.Tab("Per-class results"):
        gr.Dataframe(per_class_table(), label="Finetuned model per class (test split, highest-probability class)", interactive=False)
        gr.Dataframe(confusion_table(), label="Confusion matrix (test split)", interactive=False)
        gr.Dataframe(baseline_table(), label="Against the baselines (test split)", interactive=False)

    view_outputs = [outcome_df, headline, comparison_df, curve]
    for slider in (t_low_slider, t_high_slider):
        slider.change(operating_view, [t_low_slider, t_high_slider], view_outputs)
        slider.change(predict_image, [image_in, t_low_slider, t_high_slider], [probs_out, verdict_out])
    image_in.change(predict_image, [image_in, t_low_slider, t_high_slider], [probs_out, verdict_out])
    reset.click(lambda: (T_LOW, T_HIGH), None, [t_low_slider, t_high_slider])
    demo.load(operating_view, [t_low_slider, t_high_slider], view_outputs)

In [ ]:
# 6. Start the app (blocks until stopped). Gradio serves it on http://127.0.0.1:7860
demo.launch()